# Laya V4.3 — Repair V4.2 using the V3.1 pretrained Q/K/V + RoPE + native-window architecture

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V43_V31_Fidelity_First_Full_Then_Sliding_Colab.ipynb)

## What actually failed

The V4.2 uploaded report recorded **33.5% teacher agreement, 28.0% gold accuracy** (original: **83.5%**). The very first layer-0 replacement had only **0.445 cosine** and **2.29 NMSE** against native attention, but V4.2 forced acceptance under \`research_all\`, causing a cascade.

**V4.2 incorrectly removed the learned Q/K attention and RoPE**. Its CeNN mixed values using fixed convolution, a local average and one pooled token; this cannot represent arbitrary query-dependent attention. V3.1 retained pretrained Q/K/V, exact original RoPE, native-sized windows and offered CeNN as a small *additional* branch. Its sliding path also includes actual local SDPA, not merely a convolution.

## V4.3 corrections
- **All 10 FULL layers first, then all 18 SLIDING layers, one-by-one**, preserving 28 original ModernBERT encoder blocks, their attention/FFN residual additions, original norms and original FFNs. Only \`layer.attn\` changes.
- FULL candidate: V3.1's \`PDeltaV28FastWindowMemory\`, built from *the correct original layer*, pretrained QKV and Wo, pretrained rotary positional embeddings, an exact W1024 local/full SDPA path on ≤512-token inputs, bounded landmark/global linear memory and a CeNN convolution branch. Near-zero warm-start weight is assigned to non-native branches, avoiding the catastrophic V4.2 V-only startup.
- SLIDING candidate: V3.1's \`SlidingWindowFusion\`, pretrained QKV, RoPE, Wo, **exact native ±64 window SDPA**, with small learned CeNN conv correction. Keep its initial near-native routing.
- **Teacher-I/O input** is captured from the CURRENT accepted prefix before each layer is trained. When cosine ≥0.99 and NMSE ≤0.025 initially, **skip unnecessary distillation**. Otherwise train only necessary correction weights with a separate tiny Wo LR and probe-only checkpoint selection.
- **Default \`quality_gated\` rather than V4.2's \`research_all\`**. If teacher agreement or gold accuracy deteriorates, keep the original layer. The full-all-replacement research option is still available explicitly but is **not** a deployment claim.
- Task agreement and accuracy are calculated across the **same disjoint PROBE samples**, from a frozen native teacher. One untouched AUDIT is run after the best plan is fixed, followed by interleaved CUDA timing.
- V3.1-style attention **still uses attention** (full W1024 and sliding W129). This experiment is an accuracy restoration, not a claim of an attention-free or faster model. Genuine kernel/attention compression must be separately validated.

**New checkpoint namespace V43 and fingerprint**: old V42 adapters/progress are incompatible and not loaded. Every stage saves a resumable per-layer checkpoint; optionally mount Google Drive for persistence after runtime deletion.


In [ ]:
#@title 1. Install pinned V4.2 dependencies and GPU check
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU (T4 or better)."
print("CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0),"Torch",torch.__version__)


In [ ]:
#@title 2. Fidelity-first V3.1 attention replacement configuration
from pathlib import Path
from contextlib import contextmanager
import math,copy,json,random,hashlib,time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES,build_checkpoint_model,build_sequence,collate_items,
    normalize_question,full_attention_indices,
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv,_valid_tokens
from tinycenn_lm.sliding_window_fusion import SlidingWindowFusion
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb

SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
SEED=2043
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500 #@param {type:"integer"}
DEV_CASES=240 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
RANK=96 #@param {type:"integer"}
STEPS_FULL=240 #@param {type:"integer"}
STEPS_SLIDING=160 #@param {type:"integer"}
LOCAL_TRAIN_BATCHES=16 #@param {type:"integer"}
LOCAL_PROBE_BATCHES=4 #@param {type:"integer"}
PROBE_DECISIONS=120 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
LEARNING_RATE=0.001 #@param {type:"number"}
OUTPUT_LR=0.00001 #@param {type:"number"}
FULL_WINDOW=1024 #@param {type:"integer"}
SLIDING_RADIUS=64 #@param {type:"integer"}
SLIDING_BLOCK_SIZE=64 #@param {type:"integer"}
LANDMARK_STRIDE=16
MAX_LANDMARKS=32
FEATURE_DIM=96
TARGET_COSINE=.99
TARGET_NMSE=.025
MAX_LOCAL_NMSE=.025
LOG_EVERY=25 #@param {type:"integer"}
BENCH_REPEATS=3 #@param {type:"integer"}
SELECTION_MODE="quality_gated" #@param ["quality_gated", "research_all"]
MIN_AGREEMENT=.99 #@param {type:"number"}
MAX_GOLD_LOSS_PP=1.5 #@param {type:"number"}
REQUIRE_SPEEDUP_IN_GATED_MODE=False #@param {type:"boolean"}
RESUME=True #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}

OUT=Path("/content/Laya_V43_V31_Fidelity_First")
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/V43")
OUT.mkdir(parents=True,exist_ok=True)
assert 1<=BATCH<=4 and 16<=RANK<=256
assert 10<=STEPS_FULL and 10<=STEPS_SLIDING and LOCAL_TRAIN_BATCHES>=2
assert LOG_EVERY>=1 and BENCH_REPEATS>=2 and TRAIN_CASES+DEV_CASES<=1200
assert SELECTION_MODE in ("research_all","quality_gated")
assert FULL_WINDOW>=2*MAX_LEN-1,("V4.3 full-window teacher-parity control requires unrestricted attention",FULL_WINDOW,MAX_LEN)
assert SLIDING_RADIUS==64, "V4.3 parity control must use native radius 64"
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
        "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"
    ]
))
teacher,source_cfg=build_checkpoint_model(source_dir)
teacher.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
ORIGINAL_LAYERS=tuple(teacher.encoder.layers)
ORIGINAL_ATTN=tuple(layer.attn for layer in ORIGINAL_LAYERS)
FULL=tuple(full_attention_indices(teacher))
SLIDING=tuple(i for i in range(28) if i not in FULL)
assert FULL==(0,3,6,9,12,15,18,21,24,27)
assert len(SLIDING)==18
assert int(teacher.encoder.config.local_attention)//2==64
assert MAX_LEN<=int(source_cfg.get("max_len",MAX_LEN))
ORDER=FULL+SLIDING
print("PHASE A full:",FULL)
print("PHASE B sliding:",SLIDING)
print("One original Laya model frozen. T4 GPU",torch.cuda.get_device_name(0))
print("Selection mode:",SELECTION_MODE,"experimental forced-all" if SELECTION_MODE=="research_all" else "gated")


In [ ]:
#@title 3. Case-disjoint TRAIN / PROBE / AUDIT data + batch utilities
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V4.2 teacher helper")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items)>=max(PROBE_DECISIONS,LOCAL_PROBE_BATCHES*BATCH),"Insufficient probe decisions"
assert len(audit_items)>=AUDIT_DECISIONS,"Insufficient audit decisions"
print("official test remains untouched")


In [ ]:
#@title 4. V3.1 pretrained RoPE/QKV anchored FULL and SLIDING attention with CeNN correction

class PDeltaV28FastWindowMemory(PDelta3GDN2CLVRAttention):
    """PDelta3-inspired replacement without sequential GDN scan.

    Complexity: local attention O(T*W*d), landmark cross attention
    O(T*ceil(T/stride)*d), and kernel memory O(T*F*d).
    The landmark branch becomes quadratic if stride is fixed while T -> infinity.
    For provably bounded linear scaling, cap its count with adaptive pooling.
    """
    architecture = "pdelta_v28_fast_window_memory"

    def __init__(self, original, *, local_window=96, landmark_stride=16,
                 feature_dim=96, max_landmarks=32):
        super().__init__(
            original, feature_dim=feature_dim, conv_kernel=4,
            chunk_size=32, local_kernel=5,
            local_window=local_window, local_gate_init=0.90,
        )
        self.landmark_stride = int(landmark_stride)
        self.max_landmarks = int(max_landmarks)
        assert self.landmark_stride > 0 and self.max_landmarks > 0
        # Remove obsolete recurrent, causal-convolution and four-way fusion parameters.
        # The inherited methods _local_attention and _global_linear remain in use.
        for name in (
            "wq", "wk", "decay_w", "dt_bias", "erase_w", "erase_b",
            "write_w", "write_b", "route_proj", "route_gate_w",
            "route_gate_b", "q_conv", "k_conv", "v_conv",
            "mix_logits", "mix_gate_w", "local_gate_w",
            "local_gate_b", "log_gain",
        ):
            delattr(self, name)
        # 4.57.6 passes position_ids to ModernBERT, not position_embeddings.
        # Copy the exact pretrained per-layer RoPE instead of silently skipping it.
        self.rotary_emb = copy.deepcopy(original.rotary_emb)
        self.rotary_emb.requires_grad_(False)
        self.layer_idx = int(getattr(original, "layer_id", -1))
        h, d = self.num_heads, self.head_dim
        # [local, compressed landmarks, global linear, direct value].
        # Layer 15 is global: do not initialize with ~90% exclusively local routing.
        self.route_logits = nn.Parameter(
            torch.tensor([9.0, -5.0, -5.0, -8.0]).repeat(h, 1)
        )
        self.route_query = nn.Parameter(torch.zeros(h, 4, d))
        self.local_log_temperature = nn.Parameter(torch.zeros(h))
        self.landmark_log_temperature = nn.Parameter(torch.zeros(h))
        self.conv_gain = nn.Parameter(torch.full((h,), -12.0))
        # Pretrained QKV and Wo are copied from the teacher by BaseLayaReplacementAttention.
        # Freeze QKV and allow a very small Wo calibration.
        for p in self.Wqkv.parameters():
            p.requires_grad_(False)
        for p in self.Wo.parameters():
            p.requires_grad_(True)

    def _local_attention(self, q, k, v, valid):
        """Centered local window; avoid huge empty W×(2W-1) halos when W >= T."""
        b, h, t, d = q.shape
        w = self.local_window
        if w < t:
            return super()._local_attention(q, k, v, valid)
        allowed = valid[:, None, None, :]
        if w < 2*t-1:
            qi = torch.arange(t, device=q.device)[:, None]
            ki = torch.arange(t, device=q.device)[None, :]
            band = ((ki >= qi-(w-1)//2) & (ki <= qi+w//2))
            allowed = allowed & band[None, None]
        z = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=0.0)
        return z * valid[:, None, :, None].to(z.dtype)

    def _landmarks(self, q, k, v, valid):
        b, h, t, d = k.shape
        # Choose at most max_landmarks pooled key/value blocks. This ensures
        # O(T*M) with M <= max_landmarks at all sequence lengths.
        stride = max(self.landmark_stride, math.ceil(t / self.max_landmarks))
        blocks = math.ceil(t / stride)
        pad = blocks * stride - t
        mask = valid[:, None, :, None].to(k.dtype)
        packed_valid = F.pad(valid, (0, pad), value=False).reshape(b, blocks, stride)
        counts = packed_valid.sum(-1)
        divisor = counts.clamp_min(1).to(k.dtype)[:, None, :, None]
        kp = F.pad(k * mask, (0, 0, 0, pad)).reshape(b, h, blocks, stride, d)
        vp = F.pad(v * mask, (0, 0, 0, pad)).reshape(b, h, blocks, stride, d)
        pooled_k = kp.sum(3) / divisor
        pooled_v = vp.sum(3) / divisor
        allowed = (counts > 0)[:, None, None, :]
        temperature = self.landmark_log_temperature.clamp(-0.4, 0.4).exp()
        q_scaled = q * temperature[None, :, None, None]
        # Only T x M attention scores; never a dense T x T matrix.
        return F.scaled_dot_product_attention(
            q_scaled, pooled_k, pooled_v, attn_mask=allowed, dropout_p=0.0
        ) * mask

    def forward(self, hidden_states, position_embeddings=None,
                attention_mask=None, position_ids=None, sliding_window_mask=None,
                **kwargs):
        # Exact teacher QKV + RoPE path; apply rotary embeddings BEFORE attention.
        b, t, _ = hidden_states.shape
        qkv = self.Wqkv(hidden_states).reshape(b, t, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.transpose(3, 1).unbind(dim=2)
        if position_embeddings is None:
            cos, sin = self.rotary_emb(qkv, position_ids=position_ids)
        else:
            cos, sin = position_embeddings
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        valid = _valid_tokens(attention_mask, hidden_states)
        mask = valid[:, None, :, None].to(v.dtype)
        q, k, v = q * mask, k * mask, v * mask
        temperature = self.local_log_temperature.clamp(-0.4, 0.4).exp()
        qlocal = q * temperature[None, :, None, None]
        local = self._local_attention(qlocal, k, v, valid)
        landmarks = self._landmarks(q, k, v, valid)
        global_linear = self._global_linear(q, k, v, mask) * mask
        qnorm = F.normalize(q, dim=-1)
        dynamic = torch.einsum("bhtd,hkd->bhtk", qnorm, self.route_query)
        mixture = torch.softmax(
            self.route_logits.to(q.dtype)[None, :, None, :] + dynamic,
            dim=-1
        )
        fused = (
            mixture[..., 0:1] * local
            + mixture[..., 1:2] * landmarks
            + mixture[..., 2:3] * global_linear
            + mixture[..., 3:4] * v
        )
        local_conv = _depthwise_sequence_conv(v, self.local_weight, causal=False)
        conv_weight = torch.sigmoid(self.conv_gain).to(v.dtype)[None, :, None, None]
        fused = (fused + conv_weight * (local_conv - v)) * mask
        return self.finish(fused, hidden_states)

    def config_dict(self):
        return dict(
            architecture=self.architecture, layer_idx=self.layer_idx,
            local_window=self.local_window, landmark_stride=self.landmark_stride,
            max_landmarks=self.max_landmarks, feature_dim=self.feature_dim,
            global_linear_memory=True, local_windowed_attention=True,
            dense_global_full_attention=self.local_window >= 2 * MAX_LEN - 1, recurrent_scan=False,
            rope_applied=True, upstream_sdpa_kwargs=True,
        )


@contextmanager
def install_plan(plan):
    assert all(teacher.encoder.layers[i].attn is ORIGINAL_ATTN[i] for i in range(28))
    try:
        for i,adapter in plan.items():
            teacher.encoder.layers[i].attn=adapter
        yield teacher
    finally:
        for i in range(28):
            teacher.encoder.layers[i].attn=ORIGINAL_ATTN[i]
        assert all(teacher.encoder.layers[i].attn is ORIGINAL_ATTN[i] for i in range(28))

def new_adapter(i):
    if i in FULL:
        module=PDeltaV28FastWindowMemory(
            ORIGINAL_ATTN[i],local_window=FULL_WINDOW,
            landmark_stride=LANDMARK_STRIDE,
            feature_dim=FEATURE_DIM,max_landmarks=MAX_LANDMARKS)
        assert tuple(ORIGINAL_ATTN[i].local_attention)==(-1,-1)
    else:
        module=SlidingWindowFusion(
            ORIGINAL_ATTN[i],radius=SLIDING_RADIUS,
            block_size=SLIDING_BLOCK_SIZE)
        assert tuple(ORIGINAL_ATTN[i].local_attention)==(64,64)
    return module.to(device)

def adapter_state(adapter):
    return {k:v.detach().cpu().clone() for k,v in adapter.state_dict().items()}

def adapter_trainables(adapter):
    return [p for p in adapter.parameters() if p.requires_grad]

assert all(teacher.encoder.layers[i] is ORIGINAL_LAYERS[i] for i in range(28))
print("V3.1 kernels loaded, native full-window W",FULL_WINDOW,
      "native local-window W",2*SLIDING_RADIUS+1)
print("Real pretrained Q, K, V, rotary embedding and Wo are preserved")
print("Full CeNN non-native routing starts near zero; native route dominates")


In [ ]:
#@title 5. Native teacher targets from the CURRENT progressively compressed prefix
def cpu_float(t):
    return t.detach().float().cpu().contiguous()
def to_cuda(record):
    # clone cached tensors outside inference_mode before autograd.
    return {k:(v.to(device,non_blocking=True).clone()
               if torch.is_tensor(v) else v)
            for k,v in record.items()}

@torch.inference_mode()
def capture_layer(i,current,items,n_batches,label):
    assert i not in current
    caches=[]
    with install_plan(current):
        for j in range(n_batches):
            selected=items[j*BATCH:(j+1)*BATCH]
            if len(selected)!=BATCH:break
            b=direct_batch(selected)
            got={}
            def pre(mod,args,kwargs):
                got["x"]=args[0].detach()
                pos=kwargs.get("position_ids")
                assert pos is not None,(
                    "V3.1 RoPE needs the exact native position_ids",i)
                got["position_ids"]=pos.detach()
            def post(mod,args,kwargs,out):
                got["y"]=out[0].detach()
            h1=ORIGINAL_ATTN[i].register_forward_pre_hook(pre,with_kwargs=True)
            h2=ORIGINAL_ATTN[i].register_forward_hook(post,with_kwargs=True)
            try:
                with torch.autocast("cuda",dtype=amp_dtype):
                    teacher(**b)
            finally:
                h1.remove();h2.remove()
            assert set(got)=={"x","y","position_ids"},(i,got.keys())
            caches.append(dict(
                x=cpu_float(got["x"]),y=cpu_float(got["y"]),
                position_ids=got["position_ids"].detach().cpu().contiguous(),
                valid=b["attention_mask"].detach().cpu().bool()
            ))
    assert caches,(label,i,"empty cache")
    return caches

def mse_and_cosine(adapter,row):
    with torch.autocast("cuda",enabled=False):
        out=adapter(
            row["x"].float(),attention_mask=row["valid"],
            position_ids=row["position_ids"])[0].float()
        goal=row["y"].float()
        mask=row["valid"]
        m=mask[:,:,None].float()
        nmse=((out-goal).square()*m).sum()/(goal.square()*m).sum().clamp_min(1e-8)
        cosine=1-F.cosine_similarity(out[mask],goal[mask],dim=-1).mean()
        return nmse+.15*cosine,nmse,cosine

@torch.inference_mode()
def local_score(adapter,cache):
    adapter.eval()
    sq=power=csum=0.
    total=0
    for raw in cache:
        row=to_cuda(raw)
        with torch.autocast("cuda",dtype=amp_dtype):
            out=adapter(
                row["x"],attention_mask=row["valid"],
                position_ids=row["position_ids"])[0]
        valid=row["valid"]
        m=valid[:,:,None].float()
        sq+=float(((out.float()-row["y"].float()).square()*m).sum())
        power+=float((row["y"].float().square()*m).sum())
        csum+=float(F.cosine_similarity(
            out.float()[valid],row["y"].float()[valid],dim=-1).sum())
        total+=int(valid.sum())
    return dict(cosine=csum/max(total,1),nmse=sq/max(power,1e-8),tokens=total)

assert len(train_items)>=LOCAL_TRAIN_BATCHES*BATCH
assert len(probe_items)>=max(PROBE_DECISIONS,LOCAL_PROBE_BATCHES*BATCH)
r=capture_layer(FULL[0],{},probe_items,1,"native preflight")
assert r[0]["x"].shape==r[0]["y"].shape
assert torch.isfinite(r[0]["y"]).all()
print("Native attention capture with exact RoPE position_ids PASS")
del r


In [ ]:
#@title 6. Whole-model PROBE fidelity + interleaved CUDA speed, without AUDIT leakage
@torch.inference_mode()
def get_reference(items,limit):
    output=[]
    for start in range(0,min(limit,len(items)),BATCH):
        selected=items[start:min(start+BATCH,limit)]
        b=direct_batch(selected)
        with torch.autocast("cuda",dtype=amp_dtype):
            scores,_=teacher(**b)
        probs=torch.softmax(scores.float().masked_fill(
            ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()
        for j,item in enumerate(selected):
            output.append(dict(probs=probs[j],teacher=int(probs[j].argmax()),
                               gold=int(item["gold_index"]),
                               case=item["case_id"],qid=item["qid"]))
    return output

@torch.inference_mode()
def evaluate(plan,items,reference):
    n=len(reference)
    agree=correct=tcorrect=0;js=0.
    with install_plan(plan):
        for start in range(0,n,BATCH):
            selected=items[start:min(n,start+BATCH)]
            b=direct_batch(selected)
            with torch.autocast("cuda",dtype=amp_dtype):
                score,_=teacher(**b)
            p=torch.softmax(score.float().masked_fill(
                ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()
            for j,item in enumerate(selected):
                t=reference[start+j]["probs"];s=p[j]
                assert s.shape==t.shape
                m=(s+t)/2
                js+=float(.5*((s*(s.clamp_min(1e-8).log()-
                                    m.clamp_min(1e-8).log())).sum()+
                               (t*(t.clamp_min(1e-8).log()-
                                    m.clamp_min(1e-8).log())).sum()))
                guess=int(s.argmax())
                agree+=int(guess==reference[start+j]["teacher"])
                correct+=int(guess==item["gold_index"])
                tcorrect+=int(reference[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,
                gold_accuracy=correct/n,teacher_gold_accuracy=tcorrect/n,
                gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(before,after,repeats=BENCH_REPEATS):
    comparisons=[]
    for b in BENCH_BATCHES:
        def once(plan):
            with install_plan(plan),torch.autocast("cuda",dtype=amp_dtype):
                a=torch.cuda.Event(enable_timing=True)
                z=torch.cuda.Event(enable_timing=True)
                a.record();teacher(**b);z.record();z.synchronize()
                return float(a.elapsed_time(z))
        for _ in range(2):once(before);once(after)
        for step in range(repeats):
            readings={"a":[],"b":[]}
            order=("a","b","b","a") if step%2==0 else ("b","a","a","b")
            for label in order:
                readings[label].append(once(before if label=="a" else after))
            ma=float(np.mean(readings["a"]));mb=float(np.mean(readings["b"]))
            comparisons.append(dict(
                batch=int(b["input_ids"].shape[0]),
                tokens=int(b["input_ids"].shape[1]),
                before_ms=ma,after_ms=mb,speedup=ma/max(mb,1e-8)))
    return dict(median_speedup=float(np.median(
        [c["speedup"] for c in comparisons])),
        before_ms=float(np.median([c["before_ms"] for c in comparisons])),
        after_ms=float(np.median([c["after_ms"] for c in comparisons])),
        comparisons=comparisons)

PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
native_probe=evaluate({},probe_items,PROBE_REF)
assert native_probe["teacher_agreement"]==1 and native_probe["mean_js"]<1e-7
print("Original probe",native_probe)
print("Benchmark shapes",[(b["input_ids"].shape[0],b["input_ids"].shape[1])
                           for b in BENCH_BATCHES])


In [ ]:
#@title 7. Resume-safe experiment state and per-layer FP32 CeNN trainer
from datetime import datetime,timezone
PROGRESS_PATH=OUT/"v42_progress.json"
ADAPTER_DIR=OUT/"adapters"
ADAPTER_DIR.mkdir(parents=True,exist_ok=True)
signature=dict(
    source=SOURCE_MODEL,seed=SEED,
    train_cases=TRAIN_CASES,dev_cases=DEV_CASES,
    rank=RANK,full_window=FULL_WINDOW,sliding_radius=SLIDING_RADIUS,
    feature_dim=FEATURE_DIM,steps_full=STEPS_FULL,
    steps_sliding=STEPS_SLIDING,selection_mode=SELECTION_MODE,
    learning_rate=LEARNING_RATE,
    probe_cases=[(x["case_id"],x["qid"]) for x in probe_items[:PROBE_DECISIONS]],
    experiment_version="v43-v31-querykey-rope-a1"
)
fingerprint=hashlib.sha256(
    json.dumps(signature,sort_keys=True).encode()).hexdigest()

if RESUME and PROGRESS_PATH.exists():
    progress=json.loads(PROGRESS_PATH.read_text())
    assert progress["fingerprint"]==fingerprint,(
        "Stored progress comes from a DIFFERENT run/configuration. "
        "Change OUT or settings to start a new experiment.")
    print("Restoring completed stages:",progress["completed"])
else:
    progress=dict(fingerprint=fingerprint,completed=[],accepted=[],
                  records=[],phase_a=None,finished=False)

def save_progress():
    tmp=PROGRESS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(progress,indent=2,default=str),
                   encoding="utf-8")
    tmp.replace(PROGRESS_PATH)

def restore_adapter(i):
    saved=ADAPTER_DIR/f"attention_{i:02d}.pt"
    payload=torch.load(saved,map_location="cpu",weights_only=True)
    assert payload["index"]==i and payload["fingerprint"]==fingerprint
    model=new_adapter(i)
    model.load_state_dict(payload["weights"],strict=True)
    model.eval().requires_grad_(False)
    return model

PLAN={}
for i in progress["accepted"]:
    PLAN[int(i)]=restore_adapter(int(i))
assert progress["accepted"]==[i for i in ORDER if i in PLAN]
save_progress()
print("Restored accepted layers:",list(PLAN))
print("State file:",PROGRESS_PATH,
      "(enable SAVE_TO_GOOGLE_DRIVE for persistence across runtime resets)")

def fit_single_layer(i,prefix):
    steps=STEPS_FULL if i in FULL else STEPS_SLIDING
    train_cache=capture_layer(
        i,prefix,train_items,LOCAL_TRAIN_BATCHES,"TRAIN")
    valid_cache=capture_layer(
        i,prefix,probe_items,LOCAL_PROBE_BATCHES,"PROBE")
    model=new_adapter(i)
    opt=torch.optim.AdamW(model.parameters(),
                          lr=LEARNING_RATE,weight_decay=1e-4)
    best=None;best_score=float("inf")
    best_weights=adapter_state(model)
    record=[]
    safe_weights=adapter_state(model)
    updates=attempts=recovery=0
    while updates<steps:
        attempts+=1
        if attempts>steps+40:
            print("Stopping after repeated numerical retries at layer",i)
            break
        source=to_cuda(train_cache[(attempts-1)%len(train_cache)])
        model.train();opt.zero_grad(set_to_none=True)
        loss,nmse,cos=mse_and_cosine(model,source)
        reason=None
        if not bool(torch.isfinite(loss.detach())):
            reason="nonfinite_forward"
        else:
            loss.backward()
            if any(p.grad is not None and
                   not bool(torch.isfinite(p.grad).all())
                   for p in model.parameters()):
                reason="nonfinite_grad"
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(),.5)
                opt.step()
                if any(not bool(torch.isfinite(p).all())
                       for p in model.parameters()):
                    reason="nonfinite_parameters"
        if reason:
            model.load_state_dict(safe_weights,strict=True)
            opt.state.clear()
            for group in opt.param_groups:group["lr"]*=.5
            recovery+=1
            print(" L",i,"recover from",reason,"LR",opt.param_groups[0]["lr"])
            if recovery>=8:break
            continue
        updates+=1
        if updates%10==0:safe_weights=adapter_state(model)
        if updates==1 or updates%LOG_EVERY==0 or updates==steps:
            quality=local_score(model,valid_cache)
            score=quality["nmse"]+.15*(1-quality["cosine"])
            record.append(dict(step=updates,quality=quality,
                               train_loss=float(loss.detach())))
            if score<best_score:
                best_score=score;best=quality
                best_weights=adapter_state(model)
            print(f" L{i:02d} {updates}/{steps} "
                  f"cos={quality['cosine']:.4f} "
                  f"NMSE={quality['nmse']:.4f} "
                  f"best={best['cosine']:.4f}")
    model.load_state_dict(best_weights,strict=True)
    model.eval().requires_grad_(False)
    assert all(bool(torch.isfinite(p).all()) for p in model.parameters())
    payload=dict(index=i,fingerprint=fingerprint,
                 phase="FULL" if i in FULL else "SLIDING",
                 steps=updates,recoveries=recovery,
                 best_local=best,training_history=record,
                 weights=adapter_state(model))
    torch.save(payload,ADAPTER_DIR/f"attention_{i:02d}.pt")
    del opt,best_weights,train_cache,valid_cache
    torch.cuda.empty_cache()
    return model,payload


In [ ]:
#@title 8. PHASE A: progressively train and replace ALL TEN FULL attention modules FIRST
def make_decision(i,pre,after,paired):
    gates=dict(
        teacher_agreement=after["teacher_agreement"]>=MIN_AGREEMENT,
        gold_loss=after["gold_drop"]<=MAX_GOLD_LOSS_PP/100,
        no_step_accuracy_regression=after["gold_accuracy"]>=pre["gold_accuracy"]-.01,
        speed=(paired["median_speedup"]>1.0),
    )
    eligible=(gates["teacher_agreement"] and gates["gold_loss"] and
              gates["no_step_accuracy_regression"])
    if REQUIRE_SPEEDUP_IN_GATED_MODE:eligible=eligible and gates["speed"]
    # The user-requested full 28-module CeNN experiment is explicitly
    # allowed in research_all, while exposing every failed safety gate.
    accepted=(SELECTION_MODE=="research_all" or eligible)
    return accepted,gates

def run_stage(i,phase):
    global PLAN
    assert i in ORDER
    assert phase==("FULL" if i in FULL else "SLIDING")
    if i in progress["completed"]:
        print("SKIP completed",phase,"layer",i,
              "accepted" if i in PLAN else "native retained")
        return
    before=evaluate(PLAN,probe_items,PROBE_REF)
    print(f"\n== {phase} L{i:02d}: training native-operator CeNN targets "
          f"on current accepted prefix (accepted={len(PLAN)}) ==")
    adapter,training=fit_single_layer(i,PLAN)
    proposed={**PLAN,i:adapter}
    after=evaluate(proposed,probe_items,PROBE_REF)
    pair=benchmark_paired(PLAN,proposed)
    accepted,gates=make_decision(i,before,after,pair)
    print(" Layer",i,"probe agreement",round(after["teacher_agreement"],4),
          "gold drop",round(after["gold_drop"],4),
          "speed",round(pair["median_speedup"],4),"x",
          "accepted",accepted,"gates",gates)
    if accepted:
        PLAN[i]=adapter
    progress["records"].append(dict(
        index=i,phase=phase,accepted=accepted,
        selection_mode=SELECTION_MODE,
        quality_gates=gates,
        before_probe=before,after_probe=after,
        paired_cuda=pair,local=training["best_local"],
        train_steps=training["steps"],
        recovery_events=training["recoveries"],
    ))
    progress["completed"].append(i)
    progress["accepted"]=[j for j in ORDER if j in PLAN]
    save_progress()
    if not accepted:del adapter
    torch.cuda.empty_cache()

# STRICT ORDER. No sliding CeNN is trained before this cell finishes.
for layer_id in FULL:
    run_stage(layer_id,"FULL")
assert all(i in progress["completed"] for i in FULL)
assert not any(i in progress["completed"] for i in SLIDING) or (
    progress["phase_a"] is not None
),"Completed sliding work requires a saved full-attention phase summary."

if progress.get("phase_a") is None:
    full_probe=evaluate(PLAN,probe_items,PROBE_REF)
    full_cuda=benchmark_paired({},PLAN,repeats=max(4,BENCH_REPEATS))
    progress["phase_a"]=dict(
        replaced=[i for i in FULL if i in PLAN],
        probe=full_probe,paired_cuda_vs_native=full_cuda,
        timestamp=datetime.now(timezone.utc).isoformat())
    save_progress()
print("\nPHASE A COMPLETE. Attempted 10 FULL layers, accepted",
      len(progress["phase_a"]["replaced"]))
print("FULL-ONLY PROBE results:",progress["phase_a"]["probe"])
print("FULL-ONLY speedup:",progress["phase_a"]["paired_cuda_vs_native"]["median_speedup"])
print("Now proceed to PHASE B, sliding layers.")


In [ ]:
#@title 9. PHASE B: progressively train and replace all 18 SLIDING attention modules
assert progress.get("phase_a") is not None,(
    "Run the entire FULL-attention phase (cell 8) first.")
assert all(i in progress["completed"] for i in FULL)

for layer_id in SLIDING:
    run_stage(layer_id,"SLIDING")
assert len(progress["completed"])==28
assert progress["completed"]==list(ORDER),(
    "Completed layer ordering differs from required full-then-sliding order.",
    progress["completed"])
final_probe=evaluate(PLAN,probe_items,PROBE_REF)
progress["final_probe"]=final_probe
progress["final_plan_indices"]=[i for i in ORDER if i in PLAN]
save_progress()
print("\nPHASE B COMPLETE. All 28 attention modules individually trained/attempted.")
print("Accepted FULL",len([i for i in FULL if i in PLAN]),
      "SLIDING",len([i for i in SLIDING if i in PLAN]))
print("End of training PROBE:",final_probe)
print("ATTENTION modules modified:",
      [i for i in ORDER if i in PLAN])
print("Native residuals and original FFNs retained for ALL 28 layers.")


In [ ]:
#@title 10. ONE final untouched AUDIT and fully interleaved whole-model CUDA benchmark
assert progress.get("phase_a") and len(progress["completed"])==28
assert [i for i in ORDER if i in PLAN]==progress["accepted"]
FINAL_AUDIT_FILE=OUT/"v42_final_audit.json"
if FINAL_AUDIT_FILE.exists() and RESUME:
    prior=json.loads(FINAL_AUDIT_FILE.read_text())
    assert prior["fingerprint"]==fingerprint
    assert prior["plan_indices"]==progress["accepted"]
    final_audit=prior["audit"]
    final_speed=prior["paired_cuda"]
    final_gates=prior["gates"]
    print("Reusing final audit for IDENTICAL frozen architecture and settings.")
else:
    # Teacher reference is computed ONLY now, after PROBE-only selection.
    assert len(audit_items)>=AUDIT_DECISIONS
    audit_teacher=get_reference(audit_items,AUDIT_DECISIONS)
    assert {x["case"] for x in audit_teacher}.isdisjoint(
        {x["case"] for x in PROBE_REF})
    assert all(i in progress["completed"] for i in ORDER)
    final_audit=evaluate(PLAN,audit_items,audit_teacher)
    final_speed=benchmark_paired({},PLAN,repeats=max(6,BENCH_REPEATS))
    final_gates=dict(
        teacher_agreement_ge_target=
            final_audit["teacher_agreement"]>=MIN_AGREEMENT,
        gold_drop_le_limit=
            final_audit["gold_drop"]<=MAX_GOLD_LOSS_PP/100,
        faster_than_native=
            final_speed["median_speedup"]>1.0,
    )
    final_gates["all_pass"]=all(final_gates.values())
    FINAL_AUDIT_FILE.write_text(json.dumps(dict(
        fingerprint=fingerprint,
        plan_indices=progress["accepted"],
        audit=final_audit,paired_cuda=final_speed,
        gates=final_gates
    ),indent=2),encoding="utf-8")

print("\nFINAL original-teacher AUDIT comparison:",final_audit)
print("Final matched full-model speed:",final_speed["median_speedup"],"x")
print("Final gates:",final_gates)
if not final_gates["all_pass"]:
    print("EXPERIMENTAL ONLY. Original pretrained native Laya remains "
          "the deployment reference. Do not choose checkpoint from audit.")
else:
    print("PROVISIONAL quality and speed pass; verify across larger "
          "independent TEST samples, seeds and hardware before deployment.")


In [ ]:
#@title 11. Export adapter-only full→sliding checkpoint, per-layer ablation matrix and figures
import matplotlib.pyplot as plt

original_params=sum(p.numel() for p in teacher.parameters())
adapter_params=sum(sum(p.numel() for p in adapter.parameters())
                   for adapter in PLAN.values())
original_attn_params=sum(sum(p.numel() for p in ORIGINAL_ATTN[i].parameters())
                         for i in PLAN)
report=dict(
    version="4.2-full-first-progressive-cenn-attention-only",
    created_utc=datetime.now(timezone.utc).isoformat(),
    source_model=SOURCE_MODEL,seed=SEED,
    gpu=torch.cuda.get_device_name(0),
    original_model_parameters=original_params,
    adapter_parameters=adapter_params,
    approximated_native_attention_parameters=original_attn_params,
    original_mlp_and_residuals_preserved=True,
    original_layer_norms_preserved=True,
    source_full_attention_layers=list(FULL),
    source_sliding_layers=list(SLIDING),
    exact_experiment_order=list(ORDER),
    selection_mode=SELECTION_MODE,
    research_all_forced_acceptance=SELECTION_MODE=="research_all",
    training_steps_full=STEPS_FULL,
    training_steps_sliding=STEPS_SLIDING,
    student_rank=RANK,
    original_sliding_radius=64,
    full_CeNN_long_receptive_field=1025,
    full_training_completed_before_any_sliding=True,
    accepted_layers=progress["accepted"],
    rejected_layers=[r["index"] for r in progress["records"]
                     if not r["accepted"]],
    per_layer_training_and_probe=progress["records"],
    native_probe=native_probe,
    full_phase_probe=progress["phase_a"]["probe"],
    full_phase_cuda=progress["phase_a"]["paired_cuda_vs_native"],
    final_probe=progress["final_probe"],
    final_audit=final_audit,
    final_paired_cuda=final_speed,
    acceptance_gates=final_gates,
    official_test_unused=True,
    caveats=[
        "Only attention modules were substituted: original two residual additions "
        "and pretrained MLPs remain in the original ModernBERT layer.",
        "CeNN uses the original frozen V and Wo projections as anchors.",
        "Default research_all deliberately accepts low-fidelity experimental "
        "adapters; results MUST NOT be presented as deployment-ready.",
        "No full teacher model duplicate is allocated on the GPU.",
        "Source native attention QK weights remain resident during research; "
        "physical export/pruning is a separate step.",
        "Probe was used for acceptance; untouched audit is used only after "
        "the full plan was frozen.",
        "Convolutional RF1025 cannot in general represent arbitrary "
        "full-attention query-dependent token mixing.",
        "Checkpoint reuse requires an identical data split and config."
    ],
)
report_path=OUT/"v42_full_then_sliding_cenn_layerwise_report.json"
report_path.write_text(json.dumps(report,indent=2,default=str),
                       encoding="utf-8")
weights_path=OUT/"v42_full_then_sliding_cenn_adapters_EXPERIMENTAL.pt"
torch.save(dict(
    version=report["version"],fingerprint=fingerprint,
    source_model=SOURCE_MODEL,rank=RANK,
    phase_order=list(ORDER),accepted_indices=progress["accepted"],
    adapter_states={str(i):adapter_state(PLAN[i])
                    for i in progress["accepted"]},
    is_deployment_ready=bool(final_gates["all_pass"]),
    audit=final_audit,probe=progress["final_probe"],
),weights_path)

# Per-layer plot, distinguishing FULL and SLIDING progression.
rows=progress["records"]
indices=[r["index"] for r in rows]
acc=[100*r["after_probe"]["teacher_agreement"] for r in rows]
sp=[r["paired_cuda"]["median_speedup"] for r in rows]
plt.figure(figsize=(13,4))
plt.plot(range(len(rows)),acc,"o-",label="Probe teacher agreement (%)")
plt.axhline(100*MIN_AGREEMENT,linestyle="--",label="Agreement target")
plt.axvline(9.5,linestyle=":",label="Full → sliding phase boundary")
plt.xticks(range(len(rows)),[str(i) for i in indices])
plt.ylabel("Teacher agreement (%)")
plt.xlabel("Attention module index (first 10 full, next 18 sliding)")
plt.legend()
plt.tight_layout()
chart=OUT/"v42_progressive_agreement.png"
plt.savefig(chart,dpi=140,bbox_inches="tight")
plt.show()

print("Notebook finished.")
print("Report:",report_path)
print("Adapter-only research weights:",weights_path)
print("Full attention accepted:",len([i for i in FULL if i in PLAN]),"/10")
print("Sliding attention accepted:",len([i for i in SLIDING if i in PLAN]),"/18")
print("Final AUDIT accuracy:",final_audit["gold_accuracy"],
      "original:",final_audit["teacher_gold_accuracy"])
print("Final GPU speedup:",final_speed["median_speedup"])
print("Deployment criteria met:",final_gates["all_pass"])
